In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture
from scipy.stats import gaussian_kde
import diptest
import os

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"
OUTPUT_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

df = pd.read_csv(INPUT_CSV)

# Get all patient IDs
patients = (
    df["patient_id"]
    .dropna()
    .unique()
)

print("NUMBER OF PATIENTS FOUND:", len(patients))
print(patients)
print("="*60)

summary_results = []

for PATIENT_ID in patients:
    print("\nProcessing patient:", PATIENT_ID)

    thickness = (
        df[df["patient_id"] == PATIENT_ID]
        ["median_thickness_nm"]
        .dropna()
        .values
    )

    if len(thickness) < 5:

        print(
            "Skipping",
            PATIENT_ID,
            "- insufficient samples"
        )
        continue

    X = thickness.reshape(-1,1)

#DIP TEST

    dip_value, dip_p = diptest.diptest(
        thickness
    )

# FIT GMM MODELS
  
    models = {}
    bic_values = {}
    aic_values = {}

    for n in [1,2,3]:

        model = GaussianMixture(
            n_components=n,
            random_state=42,
            n_init=20
        )

        model.fit(X)
        models[n] = model
        bic_values[n] = model.bic(X)
        aic_values[n] = model.aic(X)

    best_components = min(
        bic_values,
        key=bic_values.get
    )
    gmm = models[best_components]


    summary_results.append({

        "patient_id": PATIENT_ID,
        "samples": len(thickness),
        "dip_statistic": dip_value,
        "dip_p_value": dip_p,
        "best_components": best_components,

        "BIC_1": bic_values[1],
        "BIC_2": bic_values[2],
        "BIC_3": bic_values[3],

        "AIC_1": aic_values[1],
        "AIC_2": aic_values[2],
        "AIC_3": aic_values[3],

        "best_BIC":
            bic_values[best_components],

        "best_AIC":
            aic_values[best_components]
    })

#GMM PLOT
    x = np.linspace(
        thickness.min()-50,
        thickness.max()+50,
        1000
    )
    x_plot = x.reshape(-1,1)

    total_density = np.exp(
        gmm.score_samples(x_plot)
    )

    plt.figure(
        figsize=(10,7)
    )

# Histogram
    plt.hist(
        thickness,
        bins=15,
        density=True,
        alpha=0.35,
        label="Observed thickness"
    )

# KDE
    kde = gaussian_kde(thickness)
    plt.plot(
        x,
        kde(x),
        linewidth=2,
        label="KDE"
    )

# Total GMM curve
    plt.plot(
        x,
        total_density,
        linewidth=3,
        label=
        f"Total GMM ({best_components} components)"
    )

# Individual Gaussian peaks
    for i in range(best_components):
        mean = gmm.means_[i][0]

        std = np.sqrt(
            gmm.covariances_[i][0][0]
        )
        weight = gmm.weights_[i]

        gaussian = (
            weight *
            (1/(std*np.sqrt(2*np.pi))) *
            np.exp(
                -0.5*((x-mean)/std)**2
            )
        )

        plt.plot(
            x,
            gaussian,
            linestyle="--",
            linewidth=2,
            label=
            f"Peak {i+1}: {mean:.1f} nm "
            f"(w={weight:.2f})"
        )

        plt.axvline(
            mean,
            linestyle=":",
            linewidth=1.5,
            label=f"Mean μ{i+1} = {mean:.1f} nm"
        )

#text box
    text = (
        f"Patient: {PATIENT_ID}\n"
        f"Samples: {len(thickness)}\n\n"
        f"Dip statistic: {dip_value:.4f}\n"
        f"Dip p-value: {dip_p:.4f}\n\n"
        f"Best GMM: {best_components} components\n"
        f"BIC: {bic_values[best_components]:.2f}\n"
        f"AIC: {aic_values[best_components]:.2f}"
    )

    plt.text(
        0.97,
        0.95,
        text,
        transform=plt.gca().transAxes,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(
            boxstyle="round",
            alpha=0.2
        )
    )

    plt.xlabel(
        "GBM thickness (nm)",
        fontsize=12
    )

    plt.ylabel(
        "Density",
        fontsize=12
    )

    plt.title(
        f"GMM Multimodality Analysis - Patient {PATIENT_ID}",
        fontsize=14,
        fontweight="bold"
    )

    plt.legend()
    plt.tight_layout()

    output_file = os.path.join(
        OUTPUT_FOLDER,
        f"GMM_patient_{PATIENT_ID}.png"
    )

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()


    print(
        "Saved:",
        output_file
    )

summary_df = pd.DataFrame(
    summary_results
)

summary_file = os.path.join(
    OUTPUT_FOLDER,
    "GMM_all_patients_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False
)

print("\n")
print("ALL PATIENT GMM ANALYSIS COMPLETED")
print("="*60)

print(
    "Patients processed:",
    len(summary_df)
)

print(
    "Summary saved:",
    summary_file
)

NUMBER OF PATIENTS FOUND: 11
['01-24' '02-24' '03-24' '04-23' '05-24' '06-24' '07-25' '08-25' '09-24'
 '10-24' '11-24']

Processing patient: 01-24
Saved: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots\GMM_patient_01-24.png

Processing patient: 02-24
Saved: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots\GMM_patient_02-24.png

Processing patient: 03-24
Saved: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots\GMM_patient_03-24.png

Processing patient: 04-23
Saved: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots\GMM_patient_04-23.png

Processing patient: 05-24
Saved: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots\GMM_patient_05-24.png

Processing patient: 06-24
Saved: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots\GMM_patient_06-24.png

Processing patient: 07-25
Saved: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots\GMM_patient_07-25.png

Processing patient: 08-25
Saved: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots\GMM_patient_08-25.png

Processing patient: 09-24
Saved: C:\Users\ishin\OneDrive\Deskto